In [ ]:
# This technique goes beyond static statistics and actually tests causality — it asks:

# If this candidate were identical in every way except for a sensitive attribute (e.g., gender or race), 
# would the model make the same decision?

# To assess whether your AI or data pipeline’s outcomes (Selected) are sensitive to 
# demographic changes when all other qualifications remain constant.

In [18]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.metrics import accuracy_score, roc_auc_score
import numpy as np

In [ ]:
# Load your dataset
df = pd.read_csv("../../datasets/recruitment.csv")

df['Age_Group'] = pd.cut(df['Age'], bins=[18, 25, 35, 45, 55, 65], labels=['18-25', '26-35', '36-45', '46-55', '56-65'])

df['Selected'] = df['Selected'].map({'Yes': 1, 'No': 0})

# Define sensitive attributes to test
sensitive_attributes = ['Gender', 'Race', 'Age_Group']

In [20]:
# Encode categorical variables
# Multi-label encoding for Skills
mlb = MultiLabelBinarizer()
skills_df = df.copy()

skills_df['Skills'] = skills_df['Skills'].fillna('').str.lower().str.split(',')

skills_encoded = pd.DataFrame(mlb.fit_transform(skills_df['Skills']),
                              columns=[f"skill_{s.replace(' ', '_')}" for s in mlb.classes_])

skills_df = pd.concat([df.drop(columns=['Skills']), skills_encoded], axis=1)

categorical_cols = ['Gender', 'Race', 'Education', 'Age_Group', "Certifications"]


df_encoded = skills_df.copy()

for col in categorical_cols:
    df_encoded[col] = pd.factorize(df_encoded[col])[0]

skills_cols = [col for col in df_encoded.columns if col.startswith('skill_')]

skills_cols = [col for col in df_encoded.columns if col.startswith('skill_')]
numerical_cols = ["Experience_Years", "Screening_Score", "Selected"]

df_encoded = df_encoded[categorical_cols + numerical_cols + skills_cols]

In [21]:
# prepare features and target
X = df_encoded.drop(columns=['Selected'])  # drop non-feature columns
y = df_encoded['Selected']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

model = LogisticRegression(max_iter=500)
model.fit(X_train, y_train)

/opt/anaconda3/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:469: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=500)

In [22]:
# Generate conterfactuals and evaluate

def flip_binary_column(df, attr):
    df_flipped = df.copy()
    unique_vals = df[attr].unique()
    if len(unique_vals) == 2:
        df_flipped[attr] = df_flipped[attr].replace({unique_vals[0]: unique_vals[1], unique_vals[1]: unique_vals[0]})
    else:
        print(f"{attr} is not binary, skipping flip.")
    return df_flipped

def model_counterfactual_audit(model, X, feature_to_flip):
    X_flipped = flip_binary_column(X, feature_to_flip)
    
    preds_original = model.predict_proba(X)[:, 1]
    preds_flipped = model.predict_proba(X_flipped)[:, 1]
    
    # Absolute difference in predicted probabilities
    diffs = np.abs(preds_original - preds_flipped)
    
    audit_result = {
        'Feature_Flipped': feature_to_flip,
        'Avg_Change': np.mean(diffs),
        'Max_Change': np.max(diffs),
        'Pct_Changed': np.mean(diffs > 0.05) * 100  # % of predictions changing > 5%
    }
    
    print(f"--- Counterfactual Audit for {feature_to_flip} ---")
    print(f"Average change in prediction: {audit_result['Avg_Change']:.3f}")
    print(f"Max change: {audit_result['Max_Change']:.3f}")
    print(f"% of candidates with ≥5% change: {audit_result['Pct_Changed']:.1f}%")
    
    return audit_result

In [24]:
sensitive_features = ['Gender', 'Age_Group', 'Race']  # Only binary or easily flippable features

audit_results = []
for feature in sensitive_features:
    result = model_counterfactual_audit(model, X_test.copy(), feature)
    audit_results.append(result)

pd.DataFrame(audit_results)


Gender is not binary, skipping flip.
--- Counterfactual Audit for Gender ---
Average change in prediction: 0.000
Max change: 0.000
% of candidates with ≥5% change: 0.0%
Age_Group is not binary, skipping flip.
--- Counterfactual Audit for Age_Group ---
Average change in prediction: 0.000
Max change: 0.000
% of candidates with ≥5% change: 0.0%
Race is not binary, skipping flip.
--- Counterfactual Audit for Race ---
Average change in prediction: 0.000
Max change: 0.000
% of candidates with ≥5% change: 0.0%


,Feature_Flipped,Avg_Change,Max_Change,Pct_Changed
0,Gender,0.0,0.0,0.0
1,Age_Group,0.0,0.0,0.0
2,Race,0.0,0.0,0.0
